# 1、TodoListMiddleware中间件

举例：

In [1]:

import truststore
from rich import print as rprint
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os

from tencentcloud.common import profile

# 从.env文件中加载环境变量
load_dotenv(override=True)
truststore.inject_into_ssl()

# CLOSEAI_API_KEY = os.getenv("CLOSEAI_API_KEY")
# CLOSEAI_BASE_URL = os.getenv("CLOSEAI_BASE_URL")
#
# model = init_chat_model(
#     model="gpt-5.4-mini",
#     model_provider="openai",
#     api_key=CLOSEAI_API_KEY,
#     base_url=CLOSEAI_BASE_URL
# )
model = init_chat_model(
    model="deepseek-v4-flash",
    model_provider="deepseek",
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url=os.getenv("DEEPSEEK_BASE_URL"),
    extra_body={"thinking": {"type": "disabled"}},
    profile={
        "max_input_tokens": 1_000_000
    }
)

In [2]:
from langchain.tools import tool
from pathlib import Path
import subprocess

WORKSPACE = Path("../todo_workspace")

@tool
def list_files(path: str = ".") -> str:
    """
    列出工作区指定目录下的文件和子目录。path 只能是相对路径。

    Args:
        path: 工作区下的相对路径，一定指向目录，默认为.，表示工作区根路径，不能访问工作区外的目录
    """
    target = (WORKSPACE / path).resolve()
    workspace_root = WORKSPACE.resolve()

    if not str(target).startswith(str(workspace_root)):
        return "错误：只允许访问工作区内的目录。"

    if not target.exists():
        return f"错误：目录不存在: {path}"

    if not target.is_dir():
        return f"错误：不是目录: {path}"

    items = sorted(target.iterdir(), key=lambda p: (p.is_file(), p.name.lower()))
    if not items:
        return f"目录为空: {path}"

    lines = []
    for item in items:
        rel = item.relative_to(workspace_root)
        kind = "[DIR]" if item.is_dir() else "[FILE]"
        lines.append(f"{kind} {rel.as_posix()}")

    return "\n".join(lines)

@tool
def read_file(path: str) -> str:
    """
    读取工作区中的文本文件内容。path 只能是相对路径。

    Args:
        path: 工作区内的文件名
    """
    file_path = (WORKSPACE / path).resolve()
    if not str(file_path).startswith(str(WORKSPACE.resolve())):
        return "错误：只允许读取工作区内的文件。"
    if not file_path.exists():
        return f"错误：文件不存在: {path}"
    return file_path.read_text(encoding="utf-8")


@tool
def write_file(path: str, content: str) -> str:
    """
    写入工作区中的文本文件。path 只能是相对路径。

    Args:
        path: 工作区内的文件名
        content: 写入文件的内容
    """
    file_path = (WORKSPACE / path).resolve()
    if not str(file_path).startswith(str(WORKSPACE.resolve())):
        return "错误：只允许写入工作区内的文件。"
    file_path.write_text(content, encoding="utf-8")
    return f"已写入文件: {path}"


@tool
def run_tests() -> str:
    """
    在工作区运行 pytest -q，并返回输出。
    不接收任何参数，返回格式为
    returncode=0|1
    STDOUT:
    STDERR:
    """
    try:
        result = subprocess.run(
            ["pytest", "-q"],
            cwd=str(WORKSPACE),
            capture_output=True,
            text=True,
            timeout=20,
        )
        return (
            f"returncode={result.returncode}\n\n"
            f"STDOUT:\n{result.stdout}\n\n"
            f"STDERR:\n{result.stderr}"
        )
    except Exception as e:
        return f"运行测试失败: {e}"


In [4]:
from langchain.agents.middleware import TodoListMiddleware
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage
from rich import print as rprint
agent = create_agent(
    model=model,
    tools=[list_files, read_file, write_file,run_tests],
    middleware=[TodoListMiddleware()],
    system_prompt="你是一个代码修复助手。遇到多步骤任务时，先使用 write_todos 制定待办事项；"
        "然后读取文件、修复代码并运行测试。工作全部在工作区下进行。"
)


response = agent.invoke({
    "messages" : [HumanMessage("请测试并修复工作区下的my_add.py文件中的代码")]
})


rprint(response)

{
    'messages': [
        HumanMessage(
            content='请测试并修复工作区下的my_add.py文件中的代码',
            additional_kwargs={},
            response_metadata={},
            id='2f740d35-3dc3-42cf-b0cd-cbbe96f497a4'
        ),
        AIMessage(
            content='让我先查看工作区的文件结构和代码内容。',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 52,
                    'prompt_tokens': 1698,
                    'total_tokens': 1750,
                    'completion_tokens_details': None,
                    'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 1664},
                    'prompt_cache_hit_tokens': 1664,
                    'prompt_cache_miss_tokens': 34
                },
                'model_provider': 'deepseek',
                'model_name': 'deepseek-v4-flash',
                'system_fingerprint': 'fp_8b330d02d0_prod0820_fp8_kvcache_20260402',
                'id': '991ad13e-ddd5-4aba-9555-414de541a76b',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--019f75c4-9f6c-75d1-bcf2-0adf5e1e71ce-0',
            tool_calls=[
                {
                    'name': 'list_files',
                    'args': {'path': '.'},
                    'id': 'call_00_jZN8xOn3guRTlNEDQHTR3444',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 1698,
                'output_tokens': 52,
                'total_tokens': 1750,
                'input_token_details': {'cache_read': 1664},
                'output_token_details': {}
            }
        ),
        ToolMessage(
            content='[DIR] __pycache__\n[FILE] my_add.py\n[FILE] test_my_add.py',
            name='list_files',
            id='f0a928f8-bf87-4030-b624-71fc3fd24059',
            tool_call_id='call_00_jZN8xOn3guRTlNEDQHTR3444'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 79,
                    'prompt_tokens': 1782,
                    'total_tokens': 1861,
                    'completion_tokens_details': None,
                    'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 1664},
                    'prompt_cache_hit_tokens': 1664,
                    'prompt_cache_miss_tokens': 118
                },
                'model_provider': 'deepseek',
                'model_name': 'deepseek-v4-flash',
                'system_fingerprint': 'fp_8b330d02d0_prod0820_fp8_kvcache_20260402',
                'id': '578a5679-6d09-49d7-94d4-ae9b63f84f71',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--019f75c4-a376-7920-b443-06732f1c921f-0',
            tool_calls=[
                {
                    'name': 'read_file',
                    'args': {'path': 'my_add.py'},
                    'id': 'call_00_40vxL3aU2qAw8ypljg567200',
                    'type': 'tool_call'
                },
                {
                    'name': 'read_file',
                    'args': {'path': 'test_my_add.py'},
                    'id': 'call_01_8VyMt1iznyJ5JT6B8Xij0133',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 1782,
                'output_tokens': 79,
                'total_tokens': 1861,
                'input_token_details': {'cache_read': 1664},
                'output_token_details': {}
            }
        ),
        ToolMessage(
            content='"""\n@Author:shkstart\n@Desc: \n"""\n\ndef add(a: int, b: int) -> int:\n    
"""返回两个整数的和"""\n    return a + b；\n',
            name='read_file',
     